# Tag analyzer eval

Uploads `data/tags/` into the Weaviate `TagPhoto` collection on EC2, where `multi2vec-clip` embeds every photo, then runs every photo in `data/test_tags/` against it. Run from the repo root or from `tag_analyzer/`; the first cell finds the root either way.

In [ ]:
import json
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tag_analyzer").is_dir())
sys.path.insert(0, str(repo_root))

import pandas as pd
from IPython.display import display
from PIL import Image

from tag_analyzer import LIBRARY_DIR, QUERIES_DIR, TEST_LABELS_FILE
from tag_analyzer import build_index, store
from tag_analyzer.match import K, THRESHOLD, match
from tag_analyzer.voting import vote

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

## 1. Upload the tag library

Drops and recreates the collection, then streams every manifest photo up as a base64 blob. Skip this cell if the library hasn't changed.

In [ ]:
build_index.main()

## 2. What Weaviate holds now

In [ ]:
client = store.connect_from_env()
collection = client.collections.get(store.COLLECTION)

brand_groups = collection.aggregate.over_all(group_by="brand").groups
library_by_brand = pd.DataFrame(
    [{"brand": group.grouped_by.value, "photos": group.total_count} for group in brand_groups]
).sort_values("brand").reset_index(drop=True)

print(f"{store.library_size(client)} photos across {len(library_by_brand)} brands")
library_by_brand

## 3. Run every test tag

Each query photo is sent to Weaviate as bytes, embedded server-side, and matched with `near_image`. The vote is the same one `match.py` uses.

In [ ]:
test_labels = json.loads(TEST_LABELS_FILE.read_text())

rows = []
neighbors_by_query = {}
for label in test_labels:
    neighbors = store.search_nearest_photos(client, QUERIES_DIR / label["filename"], K)
    neighbors_by_query[label["filename"]] = neighbors
    predicted_brand, predicted_era, confidence = vote(neighbors)
    rows.append({
        "query": label["filename"],
        "brand": label["brand"],
        "predicted_brand": predicted_brand,
        "era": label["era"],
        "predicted_era": predicted_era,
        "top_similarity": round(neighbors[0].similarity, 3),
        "confidence": round(confidence, 2),
        "brand_ok": predicted_brand.lower() == label["brand"].lower(),
        "era_ok": predicted_era.lower() == label["era"].lower(),
        "in_library": neighbors[0].similarity >= THRESHOLD,
    })

results = pd.DataFrame(rows)
results

In [ ]:
misses = results[~(results.brand_ok & results.era_ok)]

print(f"brand accuracy: {results.brand_ok.sum()}/{len(results)} = {results.brand_ok.mean():.1%}")
print(f"era accuracy:   {results.era_ok.sum()}/{len(results)} = {results.era_ok.mean():.1%}")
print(f"below THRESHOLD={THRESHOLD}: {(~results.in_library).sum()}")
misses

## 4. Look at the misses

Query photo on the left, its nearest library photos to the right, each labelled with brand, era, and similarity.

In [ ]:
THUMBNAIL_SIZE = 224


def square_thumbnail(photo_path):
    image = Image.open(photo_path).convert("RGB")
    image.thumbnail((THUMBNAIL_SIZE, THUMBNAIL_SIZE))
    canvas = Image.new("RGB", (THUMBNAIL_SIZE, THUMBNAIL_SIZE), "white")
    canvas.paste(image, ((THUMBNAIL_SIZE - image.width) // 2, (THUMBNAIL_SIZE - image.height) // 2))
    return canvas


def query_and_neighbors_strip(query_path, neighbors):
    strip = Image.new("RGB", (THUMBNAIL_SIZE * (len(neighbors) + 1), THUMBNAIL_SIZE), "white")
    strip.paste(square_thumbnail(query_path), (0, 0))
    for position, neighbor in enumerate(neighbors, start=1):
        strip.paste(square_thumbnail(LIBRARY_DIR / neighbor.filename), (THUMBNAIL_SIZE * position, 0))
    return strip


def describe_neighbors(neighbors):
    return "  |  ".join(f"{neighbor.brand}/{neighbor.era} {neighbor.similarity:.3f}" for neighbor in neighbors)


for _, row in misses.iterrows():
    neighbors = neighbors_by_query[row.query]
    print(f"{row.query}: expected {row.brand}/{row.era}, got {row.predicted_brand}/{row.predicted_era}")
    print(f"   neighbors: {describe_neighbors(neighbors)}")
    display(query_and_neighbors_strip(QUERIES_DIR / row.query, neighbors))

## 5. Try any photo

Change `photo` to any path. This goes through `match.py`, so the `THRESHOLD` check applies and `in_library` can come back `False`.

In [ ]:
photo = sorted(QUERIES_DIR.iterdir())[0]

result = match(client, photo)
print(result.model_dump_json(indent=2))
display(query_and_neighbors_strip(photo, result.neighbors))

In [ ]:
client.close()